In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")
print(f"  pandas version  : {pd.__version__}")
print(f"  numpy version   : {np.__version__}")
print(f"  sqlalchemy      : imported")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Mansoor\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\Mansoor\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\Mansoor\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 711, in start
    self.io_loop.start()
  File "C:\Users\Mansoor\anaconda3\Lib\si

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Mansoor\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\Mansoor\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\Mansoor\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 711, in start
    self.io_loop.start()
  File "C:\Users\Mansoor\anaconda3\Lib\si

AttributeError: _ARRAY_API not found

✓ Libraries imported successfully
  pandas version  : 3.0.2
  numpy version   : 2.4.4
  sqlalchemy      : imported


In [2]:
load_dotenv()

DB_URI = os.getenv("SUPABASE_DB_URI")

if not DB_URI:
    raise ValueError("SUPABASE_DB_URI not found in .env file. Please check your .env setup.")

engine = create_engine(DB_URI)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user, version();"))
    row = result.fetchone()
    print("✓ Connected to Supabase successfully")
    print(f"  Database : {row[0]}")
    print(f"  User     : {row[1]}")
    print(f"  Version  : {row[2][:50]}")

✓ Connected to Supabase successfully
  Database : postgres
  User     : postgres
  Version  : PostgreSQL 17.6 on aarch64-unknown-linux-gnu, comp


In [3]:
import pandas as pd
import sqlalchemy
print(pd.__version__)
print(sqlalchemy.__version__)

3.0.2
2.0.49


In [4]:
from sqlalchemy import text

tables = {
    "orders"       : "olist_orders",
    "order_items"  : "olist_order_items",
    "customers"    : "olist_customers",
    "sellers"      : "olist_sellers",
    "products"     : "olist_products",
    "payments"     : "olist_order_payments",
    "reviews"      : "olist_order_reviews",
    "geolocation"  : "olist_geolocation",
    "cat_translate": "product_category_translation",
}

dfs = {}
with engine.connect() as conn:
    for key, table in tables.items():
        dfs[key] = pd.read_sql(text(f"SELECT * FROM {table}"), conn)
        print(f"  ✓ {table:<40} → shape: {dfs[key].shape}")

# Unpack into individual DataFrames for clarity
df_orders    = dfs["orders"]
df_items     = dfs["order_items"]
df_customers = dfs["customers"]
df_sellers   = dfs["sellers"]
df_products  = dfs["products"]
df_payments  = dfs["payments"]
df_reviews   = dfs["reviews"]
df_geo       = dfs["geolocation"]
df_cat       = dfs["cat_translate"]

print("\n✓ All tables loaded into individual DataFrames")

  ✓ olist_orders                             → shape: (99441, 8)
  ✓ olist_order_items                        → shape: (112650, 7)
  ✓ olist_customers                          → shape: (99441, 5)
  ✓ olist_sellers                            → shape: (3095, 4)
  ✓ olist_products                           → shape: (32951, 9)
  ✓ olist_order_payments                     → shape: (103886, 5)
  ✓ olist_order_reviews                      → shape: (99224, 7)
  ✓ olist_geolocation                        → shape: (1000163, 5)
  ✓ product_category_translation             → shape: (71, 2)

✓ All tables loaded into individual DataFrames


In [5]:
for name, df in zip(tables.keys(), dfs.values()):
    print(f"\n{'='*55}")
    print(f"  TABLE: {name}")
    print(f"{'='*55}")
    print(df.dtypes.to_string())


  TABLE: orders
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str

  TABLE: order_items
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64

  TABLE: customers
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str

  TABLE: sellers
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str

  TABLE: products
product_id                        str
product_category_name             str
product_name_leng

In [6]:
print("NULL COUNTS PER TABLE\n")
for name, df in zip(tables.keys(), dfs.values()):
    null_counts = df.isnull().sum()
    nulls_exist = null_counts[null_counts > 0]
    print(f"\n  [{name}]")
    if nulls_exist.empty:
        print("    → No nulls found")
    else:
        for col, cnt in nulls_exist.items():
            pct = cnt / len(df) * 100
            print(f"    {col:<45} {cnt:>6} nulls  ({pct:.2f}%)")

NULL COUNTS PER TABLE


  [orders]
    order_approved_at                                160 nulls  (0.16%)
    order_delivered_carrier_date                    1783 nulls  (1.79%)
    order_delivered_customer_date                   2965 nulls  (2.98%)

  [order_items]
    → No nulls found

  [customers]
    → No nulls found

  [sellers]
    → No nulls found

  [products]
    product_category_name                            610 nulls  (1.85%)
    product_name_lenght                              610 nulls  (1.85%)
    product_description_lenght                       610 nulls  (1.85%)
    product_photos_qty                               610 nulls  (1.85%)
    product_weight_g                                   2 nulls  (0.01%)
    product_length_cm                                  2 nulls  (0.01%)
    product_height_cm                                  2 nulls  (0.01%)
    product_width_cm                                   2 nulls  (0.01%)

  [payments]
    → No nulls found

  [reviews]
   

In [7]:
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in timestamp_cols:
    df_orders[col] = pd.to_datetime(df_orders[col], errors="coerce")

df_items["shipping_limit_date"] = pd.to_datetime(
    df_items["shipping_limit_date"], errors="coerce"
)

print("✓ Timestamp columns converted")
print("\n  df_orders dtypes after conversion:")
print(df_orders[timestamp_cols].dtypes.to_string())
print(f"\n  shipping_limit_date dtype : {df_items['shipping_limit_date'].dtype}")

✓ Timestamp columns converted

  df_orders dtypes after conversion:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]

  shipping_limit_date dtype : datetime64[us]


In [8]:
print(f"  Total orders before filter : {len(df_orders):,}")
print(f"\n  order_status value counts:")
print(df_orders["order_status"].value_counts().to_string())

df_orders_delivered = df_orders[
    df_orders["order_status"] == "delivered"
].copy()

# Drop rows where either delivery timestamp is null
df_orders_delivered = df_orders_delivered.dropna(
    subset=["order_delivered_customer_date", "order_estimated_delivery_date"]
)

print(f"\n  Orders after 'delivered' filter + null drop : {len(df_orders_delivered):,}")
print(f"  Dropped : {len(df_orders) - len(df_orders_delivered):,} rows")

  Total orders before filter : 99,441

  order_status value counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2

  Orders after 'delivered' filter + null drop : 96,470
  Dropped : 2,971 rows


In [9]:
df = df_orders_delivered.merge(
    df_customers,
    on="customer_id",
    how="left"
)

print(f"  After merging customers   → shape: {df.shape}")
print(f"  Null customer_state       : {df['customer_state'].isnull().sum()}")

  After merging customers   → shape: (96470, 12)
  Null customer_state       : 0


In [10]:
# Multiple items per order → aggregate to order level
items_agg = df_items.groupby("order_id").agg(
    item_count          = ("order_item_id", "count"),
    total_price         = ("price", "sum"),
    total_freight       = ("freight_value", "sum"),
    avg_price           = ("price", "mean"),
    unique_sellers      = ("seller_id", "nunique"),
    unique_products     = ("product_id", "nunique"),
    primary_seller_id   = ("seller_id", "first"),   # seller of first item
    primary_product_id  = ("product_id", "first"),  # product of first item
).reset_index()

print(f"  items_agg shape : {items_agg.shape}")
print(f"\n  Sample:")
print(items_agg.head(3).to_string())

df = df.merge(items_agg, on="order_id", how="left")
print(f"\n  After merging order items → shape: {df.shape}")

  items_agg shape : (98666, 9)

  Sample:
                           order_id  item_count  total_price  total_freight  avg_price  unique_sellers  unique_products                 primary_seller_id                primary_product_id
0  00010242fe8c5a6d1ba2dd792cb16214           1         58.9          13.29       58.9               1                1  48436dade18ac8b2bce089ec2a041202  4244733e06e7ecb4970a6e2683c13e61
1  00018f77f2f0320c557190d7a144bdd3           1        239.9          19.93      239.9               1                1  dd7ddc04e1b6c2c614352b383efe2d36  e5f2d52b802189ee658865ca93d83a8f
2  000229ec398224ef6ca0657da4fc703e           1        199.0          17.87      199.0               1                1  5b51032eddd242adc84c38acab88f23d  c777355d18b72b67abbeef9df44fd0fd

  After merging order items → shape: (96470, 20)


In [11]:
df = df.merge(
    df_sellers.rename(columns={
        "seller_id"              : "primary_seller_id",
        "seller_zip_code_prefix" : "seller_zip_code_prefix",
        "seller_city"            : "seller_city",
        "seller_state"           : "seller_state",
    }),
    on="primary_seller_id",
    how="left"
)

print(f"  After merging sellers     → shape: {df.shape}")
print(f"  Null seller_state         : {df['seller_state'].isnull().sum()}")

  After merging sellers     → shape: (96470, 23)
  Null seller_state         : 0


In [12]:
df = df.merge(
    df_products.rename(columns={"product_id": "primary_product_id"}),
    on="primary_product_id",
    how="left"
)

print(f"  After merging products    → shape: {df.shape}")
print(f"  Null product_category     : {df['product_category_name'].isnull().sum()}")

  After merging products    → shape: (96470, 31)
  Null product_category     : 1359


In [13]:
df = df.merge(
    df_cat,
    on="product_category_name",
    how="left"
)

print(f"  After merging category translation → shape: {df.shape}")
print(f"  Null product_category_english      : {df['product_category_name_english'].isnull().sum()}")

  After merging category translation → shape: (96470, 32)
  Null product_category_english      : 1378


In [14]:
payments_agg = df_payments.groupby("order_id").agg(
    total_payment_value   = ("payment_value", "sum"),
    payment_installments  = ("payment_installments", "max"),
    payment_methods_used  = ("payment_type", "nunique"),
    primary_payment_type  = ("payment_type", "first"),
).reset_index()

print(f"  payments_agg shape : {payments_agg.shape}")

df = df.merge(payments_agg, on="order_id", how="left")
print(f"  After merging payments    → shape: {df.shape}")

  payments_agg shape : (99440, 5)
  After merging payments    → shape: (96470, 36)


In [15]:
reviews_agg = df_reviews.groupby("order_id").agg(
    review_score           = ("review_score", "mean"),
    review_count           = ("review_id", "count"),
    has_review_comment     = ("review_comment_message", lambda x: int(x.notna().any())),
).reset_index()

print(f"  reviews_agg shape : {reviews_agg.shape}")

df = df.merge(reviews_agg, on="order_id", how="left")
print(f"  After merging reviews     → shape: {df.shape}")

  reviews_agg shape : (98673, 4)
  After merging reviews     → shape: (96470, 39)


In [16]:
print("=" * 60)
print("  MASTER DATAFRAME SUMMARY")
print("=" * 60)
print(f"  Shape            : {df.shape}")
print(f"  Rows             : {df.shape[0]:,}")
print(f"  Columns          : {df.shape[1]:,}")
print(f"\n  Column List:")
for i, col in enumerate(df.columns, 1):
    print(f"    {i:>2}. {col}")

  MASTER DATAFRAME SUMMARY
  Shape            : (96470, 39)
  Rows             : 96,470
  Columns          : 39

  Column List:
     1. order_id
     2. customer_id
     3. order_status
     4. order_purchase_timestamp
     5. order_approved_at
     6. order_delivered_carrier_date
     7. order_delivered_customer_date
     8. order_estimated_delivery_date
     9. customer_unique_id
    10. customer_zip_code_prefix
    11. customer_city
    12. customer_state
    13. item_count
    14. total_price
    15. total_freight
    16. avg_price
    17. unique_sellers
    18. unique_products
    19. primary_seller_id
    20. primary_product_id
    21. seller_zip_code_prefix
    22. seller_city
    23. seller_state
    24. product_category_name
    25. product_name_lenght
    26. product_description_lenght
    27. product_photos_qty
    28. product_weight_g
    29. product_length_cm
    30. product_height_cm
    31. product_width_cm
    32. product_category_name_english
    33. total_payment_valu

In [17]:
null_summary = df.isnull().sum()
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)

print("  NULL COUNTS IN MASTER DATAFRAME (columns with nulls only)\n")
for col, cnt in null_summary.items():
    pct = cnt / len(df) * 100
    print(f"    {col:<45} {cnt:>6}  ({pct:.2f}%)")

print(f"\n  Total columns with nulls : {len(null_summary)}")
print(f"  Total columns            : {df.shape[1]}")

  NULL COUNTS IN MASTER DATAFRAME (columns with nulls only)

    product_category_name_english                   1378  (1.43%)
    product_category_name                           1359  (1.41%)
    product_photos_qty                              1359  (1.41%)
    product_name_lenght                             1359  (1.41%)
    product_description_lenght                      1359  (1.41%)
    review_score                                     646  (0.67%)
    has_review_comment                               646  (0.67%)
    review_count                                     646  (0.67%)
    product_width_cm                                  16  (0.02%)
    product_height_cm                                 16  (0.02%)
    product_length_cm                                 16  (0.02%)
    product_weight_g                                  16  (0.02%)
    order_approved_at                                 14  (0.01%)
    order_delivered_carrier_date                       1  (0.00%)
    payment_met

In [18]:
print("SANITY CHECKS\n")

# 1. Duplicate order IDs
dup_orders = df["order_id"].duplicated().sum()
print(f"  Duplicate order_ids         : {dup_orders}")

# 2. Timestamp ordering check — purchase should always be before delivery
bad_timestamps = (
    df["order_purchase_timestamp"] > df["order_delivered_customer_date"]
).sum()
print(f"  Purchase after delivery     : {bad_timestamps}")

# 3. Estimated delivery before purchase
bad_estimated = (
    df["order_purchase_timestamp"] > df["order_estimated_delivery_date"]
).sum()
print(f"  Estimated delivery < purchase : {bad_estimated}")

# 4. Negative freight
neg_freight = (df["total_freight"] < 0).sum()
print(f"  Negative freight_value rows : {neg_freight}")

# 5. Negative price
neg_price = (df["total_price"] < 0).sum()
print(f"  Negative price rows         : {neg_price}")

# 6. Date range
print(f"\n  Purchase date range:")
print(f"    Min : {df['order_purchase_timestamp'].min()}")
print(f"    Max : {df['order_purchase_timestamp'].max()}")

SANITY CHECKS

  Duplicate order_ids         : 0
  Purchase after delivery     : 0
  Estimated delivery < purchase : 0
  Negative freight_value rows : 0
  Negative price rows         : 0

  Purchase date range:
    Min : 2016-09-15 12:16:38
    Max : 2018-08-29 15:00:37


In [19]:
import os
os.makedirs("../data/processed", exist_ok=True)

save_path = "../data/processed/master_df.parquet"
df.to_parquet(save_path, index=False)

print(f"✓ Master DataFrame saved to: {save_path}")
print(f"  Shape  : {df.shape}")
print(f"  Size   : {os.path.getsize(save_path) / 1024 / 1024:.2f} MB")

✓ Master DataFrame saved to: ../data/processed/master_df.parquet
  Shape  : (96470, 39)
  Size   : 16.82 MB
